# Predictive Modeling

The purpose of this notebook is to develope machine learning models capable of predicting CMS Hospital Overall Ratings using hospital characteristics, patient experience measures, infection metrics, mortality measures, and readmission measures derived during exploratory data analysis and feature engineering.

Three supervised machine learning algorithms will be developed and compared:
    -Logistic Regression
    -Decision Tree
    -Random Forest

The resulting models will be evaluated in 05_model_evaluation.ipynb.

# Imports

In [30]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Load the Master Dataset

In [31]:
ROOT_DIR = Path.cwd().parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

master_df = pd.read_csv(
    PROCESSED_DIR / "master_dataset.csv"
)

print(master_df.shape)
display(master_df.head())

(5432, 63)


,facility_id,facility_name,address,city_town,state,zip_code,county_parish,telephone_number,hospital_type,hospital_ownership,...,copd_mortality,heart_failure_mortality,pneumonia_mortality,stroke_mortality,heart_attack_readmit,cabg_readmit,copd_readmit,heart_failure_readmit,hip_knee_readmit,pneumonia_readmit
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,9.4,10.2,18.4,13.5,13.0,10.1,18.0,20.1,4.8,16.0
1,10005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,8.9,14.1,21.2,12.9,NaN,NaN,17.1,19.8,4.2,13.9
2,10006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,8.7,12.5,19.6,12.4,12.5,10.6,19.1,19.5,5.1,15.7
3,10007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,11.2,13.4,25.4,NaN,NaN,NaN,18.6,20.9,NaN,16.5
4,10011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,9.1,14.4,20.1,12.8,13.2,11.9,18.2,20.9,NaN,16.2


# Confirm values of Master Dataset

In [32]:
master_df["hospital_overall_rating"].value_counts(dropna=False).sort_index()

hospital_overall_rating
1.0     199
2.0     662
3.0     987
4.0     950
5.0     384
NaN    2250
Name: count, dtype: int64

# Remove rows with missing target value (Overall Rating/NaN)
Removing rows with NaN ensures better results by only focusing on rows containing the target value. 

In [33]:
model_df = master_df.dropna(subset=["hospital_overall_rating"]).copy()

print(model_df.shape)

(3182, 63)


# Identify features

In [34]:
model_df.info()

<class 'pandas.DataFrame'>
Index: 3182 entries, 0 to 5408
Data columns (total 63 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   facility_id                                       3182 non-null   str    
 1   facility_name                                     3182 non-null   str    
 2   address                                           3182 non-null   str    
 3   city_town                                         3182 non-null   str    
 4   state                                             3182 non-null   str    
 5   zip_code                                          3182 non-null   int64  
 6   county_parish                                     3182 non-null   str    
 7   telephone_number                                  3182 non-null   str    
 8   hospital_type                                     3182 non-null   str    
 9   hospital_ownership                 

In [35]:
model_df.head()

,facility_id,facility_name,address,city_town,state,zip_code,county_parish,telephone_number,hospital_type,hospital_ownership,...,copd_mortality,heart_failure_mortality,pneumonia_mortality,stroke_mortality,heart_attack_readmit,cabg_readmit,copd_readmit,heart_failure_readmit,hip_knee_readmit,pneumonia_readmit
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,9.4,10.2,18.4,13.5,13.0,10.1,18.0,20.1,4.8,16.0
1,10005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,8.9,14.1,21.2,12.9,NaN,NaN,17.1,19.8,4.2,13.9
2,10006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,8.7,12.5,19.6,12.4,12.5,10.6,19.1,19.5,5.1,15.7
3,10007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,11.2,13.4,25.4,NaN,NaN,NaN,18.6,20.9,NaN,16.5
4,10011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,9.1,14.4,20.1,12.8,13.2,11.9,18.2,20.9,NaN,16.2


# Convert float values to integers
This will allow scikit-learn to treat them as labels.

In [36]:
model_df["hospital_overall_rating"] = (
    model_df["hospital_overall_rating"].astype(int)
)

model_df["hospital_overall_rating"].value_counts().sort_index()

hospital_overall_rating
1    199
2    662
3    987
4    950
5    384
Name: count, dtype: int64

# Remove unnecessary columns
Remove columns that should not be included as predictors.

In [37]:
columns_to_drop = [
    "facility_id",
    "facility_name",
    "address",
    "city_town",
    "telephone_number",
    "zip_code",
    "hospital_overall_rating",          # target (remove from X only)
    "hospital_overall_rating_footnote",
    "mort_group_footnote",
]

In [38]:
print(model_df.columns.tolist())

['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'mort_group_footnote', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'safety_group_footnote', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'readm_group_footnote', 'pt_exp_group_measure_count', 'count_of_facility_pt_exp_measures', 'pt_exp_group_footnote', 'te_group_measure_count', 'co

In [39]:
target = "hospital_overall_rating"

In [40]:
categorical_features = [
    "city_town",
    "state",
    "county_parish",
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "meets_criteria_for_birthing_friendly_designation",
]

In [41]:
summary_columns = [
    "mort_group_measure_count",
    "count_of_facility_mort_measures",
    "count_of_mort_measures_better",
    "count_of_mort_measures_no_different",
    "count_of_mort_measures_worse",

    "safety_group_measure_count",
    "count_of_facility_safety_measures",
    "count_of_safety_measures_better",
    "count_of_safety_measures_no_different",
    "count_of_safety_measures_worse",

    "readm_group_measure_count",
    "count_of_facility_readm_measures",
    "count_of_readm_measures_better",
    "count_of_readm_measures_no_different",
    "count_of_readm_measures_worse",

    "pt_exp_group_measure_count",
    "count_of_facility_pt_exp_measures",

    "te_group_measure_count",
    "count_of_facility_te_measures",

    "mort_group_footnote",
    "safety_group_footnote",
    "readm_group_footnote",
    "pt_exp_group_footnote",
    "te_group_footnote",
]

In [42]:
numeric_features = [
    "heart_attack_readmit",
    "cabg_readmit",
    "copd_readmit",
    "heart_failure_readmit",
    "hip_knee_readmit",
    "pneumonia_readmit",
]

# Define columns to drop and define Summary Columns

In [43]:
drop_columns = [
    "facility_id",
    "facility_name",
    "address",
    "telephone_number",
    "hospital_overall_rating",
    "hospital_overall_rating_footnote",
    "zip_code",
]

In [44]:
summary_columns = [
    "mort_group_measure_count",
    "count_of_facility_mort_measures",
    "count_of_mort_measures_better",
    "count_of_mort_measures_no_different",
    "count_of_mort_measures_worse",

    "safety_group_measure_count",
    "count_of_facility_safety_measures",
    "count_of_safety_measures_better",
    "count_of_safety_measures_no_different",
    "count_of_safety_measures_worse",

    "readm_group_measure_count",
    "count_of_facility_readm_measures",
    "count_of_readm_measures_better",
    "count_of_readm_measures_no_different",
    "count_of_readm_measures_worse",

    "pt_exp_group_measure_count",
    "count_of_facility_pt_exp_measures",

    "te_group_measure_count",
    "count_of_facility_te_measures",

    "mort_group_footnote",
    "safety_group_footnote",
    "readm_group_footnote",
    "pt_exp_group_footnote",
    "te_group_footnote",
]

# Build X & Y 
I've cleaned the columns and identified my target and categories. Next step builds the x & y models.

In [45]:
target = "hospital_overall_rating"

exclude = drop_columns + summary_columns

X = model_df.drop(columns=exclude)
y = model_df[target].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

X shape: (3182, 32)
y shape: (3182,)


,city_town,state,county_parish,hospital_type,hospital_ownership,emergency_services,meets_criteria_for_birthing_friendly_designation,cleanliness_star,nurse_comm_star,doctor_comm_star,...,copd_mortality,heart_failure_mortality,pneumonia_mortality,stroke_mortality,heart_attack_readmit,cabg_readmit,copd_readmit,heart_failure_readmit,hip_knee_readmit,pneumonia_readmit
0,DOTHAN,AL,HOUSTON,Acute Care Hospitals,Government - Hospital District or Authority,Yes,Y,3.0,3.0,4.0,...,9.4,10.2,18.4,13.5,13.0,10.1,18.0,20.1,4.8,16.0
1,BOAZ,AL,MARSHALL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,Y,3.0,3.0,4.0,...,8.9,14.1,21.2,12.9,NaN,NaN,17.1,19.8,4.2,13.9
2,FLORENCE,AL,LAUDERDALE,Acute Care Hospitals,Proprietary,Yes,Y,2.0,2.0,2.0,...,8.7,12.5,19.6,12.4,12.5,10.6,19.1,19.5,5.1,15.7
3,OPP,AL,COVINGTON,Acute Care Hospitals,Voluntary non-profit - Private,Yes,NaN,NaN,NaN,NaN,...,11.2,13.4,25.4,NaN,NaN,NaN,18.6,20.9,NaN,16.5
4,BIRMINGHAM,AL,JEFFERSON,Acute Care Hospitals,Voluntary non-profit - Private,Yes,NaN,2.0,3.0,3.0,...,9.1,14.4,20.1,12.8,13.2,11.9,18.2,20.9,NaN,16.2


In [46]:
[col for col in model_df.columns if "star" in col.lower()]

['cleanliness_star',
 'nurse_comm_star',
 'doctor_comm_star',
 'medicine_comm_star',
 'discharge_info_star',
 'quietness_star',
 'recommend_hospital_star']

# Build Feature Matrix

In [47]:
# Keep only hospitals with a CMS Overall Rating
model_df = master_df.dropna(
    subset=["hospital_overall_rating"]
).copy()

# Convert the target to integer class labels
model_df["hospital_overall_rating"] = (
    model_df["hospital_overall_rating"].astype(int)
)

print(model_df.shape)

display(
    model_df["hospital_overall_rating"]
    .value_counts()
    .sort_index()
)

(3182, 63)


hospital_overall_rating
1    199
2    662
3    987
4    950
5    384
Name: count, dtype: int64

In [48]:
target = "hospital_overall_rating"

drop_columns = [
    # identifiers
    "facility_id",
    "facility_name",
    "address",
    "telephone_number",
    "zip_code",

    # target
    "hospital_overall_rating",

    # footnotes
    "hospital_overall_rating_footnote",
    "mort_group_footnote",
    "safety_group_footnote",
    "readm_group_footnote",
    "pt_exp_group_footnote",
    "te_group_footnote",

    # CMS summary variables
    "mort_group_measure_count",
    "count_of_facility_mort_measures",
    "count_of_mort_measures_better",
    "count_of_mort_measures_no_different",
    "count_of_mort_measures_worse",

    "safety_group_measure_count",
    "count_of_facility_safety_measures",
    "count_of_safety_measures_better",
    "count_of_safety_measures_no_different",
    "count_of_safety_measures_worse",

    "readm_group_measure_count",
    "count_of_facility_readm_measures",
    "count_of_readm_measures_better",
    "count_of_readm_measures_no_different",
    "count_of_readm_measures_worse",

    "pt_exp_group_measure_count",
    "count_of_facility_pt_exp_measures",

    "te_group_measure_count",
    "count_of_facility_te_measures",
]

X = model_df.drop(columns=drop_columns)
y = model_df[target].astype(int)

print("Predictors:", X.shape)
print("Target:", y.shape)

Predictors: (3182, 32)
Target: (3182,)


# Identify Feature Types

In [49]:
categorical_features = (
    X.select_dtypes(include=["object", "string"])
    .columns
    .tolist()
)

numeric_features = (
    X.select_dtypes(include=["number"])
    .columns
    .tolist()
)

print(f"Categorical features ({len(categorical_features)}):")
print(categorical_features)

print(f"\nNumeric features ({len(numeric_features)}):")
print(numeric_features)

Categorical features (7):
['city_town', 'state', 'county_parish', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation']

Numeric features (25):
['cleanliness_star', 'nurse_comm_star', 'doctor_comm_star', 'medicine_comm_star', 'discharge_info_star', 'quietness_star', 'recommend_hospital_star', 'clabsi_sir', 'cauti_sir', 'colon_ssi_sir', 'hysterectomy_ssi_sir', 'mrsa_sir', 'cdiff_sir', 'heart_attack_mortality', 'cabg_mortality', 'copd_mortality', 'heart_failure_mortality', 'pneumonia_mortality', 'stroke_mortality', 'heart_attack_readmit', 'cabg_readmit', 'copd_readmit', 'heart_failure_readmit', 'hip_knee_readmit', 'pneumonia_readmit']


# Build Preprocessing Pipeline
This pipeline will replace missing values and standardize values for numeric predictors. For categorical predictors, it will replace missing values with most frequent, convert categories into one-hot encoded columns and will safely handle categories that may appear in test data but not training data.

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [51]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
    ]
)

# Split Data

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training predictors:", X_train.shape)
print("Testing predictors:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training predictors: (2545, 32)
Testing predictors: (637, 32)
Training target: (2545,)
Testing target: (637,)


# Train Model using Logistic Regression

In [53]:
from sklearn.linear_model import LogisticRegression

In [54]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

In [55]:
logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.


# Generate predictions

In [56]:
logistic_predictions = logistic_pipeline.predict(X_test)

print("Predictions generated:", len(logistic_predictions))

Predictions generated: 637


# Preliminary accuracy check

In [57]:
from sklearn.metrics import accuracy_score

logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions,
)

print(
    f"Preliminary Logistic Regression accuracy: "
    f"{logistic_accuracy:.3f}"
)

Preliminary Logistic Regression accuracy: 0.537


# Save preliminary test predictions

In [58]:
logistic_predictions = logistic_pipeline.predict(X_test)

print("Predictions generated:", len(logistic_predictions))
print("First 10 predictions:", logistic_predictions[:10])

Predictions generated: 637
First 10 predictions: [5 2 4 5 2 4 4 3 2 4]


# Preliminary Results
The Logistic Regression model successfully trained using the preprocessed training dataset and generated predictions for the testing dataset. The preliminary accuracy on the testing data was **53.7%**.

This model serves as the baseline for comparison with more flexible machine learning algorithms such as Decision Trees and Random Forests. A complete evaluation, including precision, recall, F1-score, confusion matrices, and model comparisons, will be presented in the Model Evaluation notebook.

# Decision Tree

In [59]:
from sklearn.tree import DecisionTreeClassifier

In [60]:
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42,
                max_depth=10,
                min_samples_split=10,
                min_samples_leaf=5,
            ),
        ),
    ]
)

In [61]:
decision_tree_pipeline.fit(X_train, y_train)

print("Decision Tree training complete.")

Decision Tree training complete.


In [62]:
decision_tree_predictions = decision_tree_pipeline.predict(X_test)

In [63]:
decision_tree_accuracy = accuracy_score(
    y_test,
    decision_tree_predictions,
)

print(
    f"Preliminary Decision Tree accuracy: "
    f"{decision_tree_accuracy:.3f}"
)

Preliminary Decision Tree accuracy: 0.451


In [64]:
model_results = []

model_results.append(
    {
        "Model": "Decision Tree",
        "Accuracy": decision_tree_accuracy,
    }
)

# Random Forest

In [65]:
from sklearn.ensemble import RandomForestClassifier

In [66]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_split=5,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [67]:
random_forest_pipeline.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


In [68]:
random_forest_predictions = random_forest_pipeline.predict(X_test)

print("Predictions generated:", len(random_forest_predictions))
print("First 10 predictions:", random_forest_predictions[:10])

Predictions generated: 637
First 10 predictions: [5 1 3 5 1 4 5 2 2 5]


In [69]:
random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions,
)

print(
    f"Preliminary Random Forest accuracy: "
    f"{random_forest_accuracy:.3f}"
)

Preliminary Random Forest accuracy: 0.418


In [70]:
model_results.append(
    {
        "Model": "Random Forest",
        "Accuracy": random_forest_accuracy,
    }
)

In [71]:
model_results = [
    {
        "Model": "Logistic Regression",
        "Accuracy": logistic_accuracy,
    },
    {
        "Model": "Decision Tree",
        "Accuracy": decision_tree_accuracy,
    },
    {
        "Model": "Random Forest",
        "Accuracy": random_forest_accuracy,
    },
]

model_results_df = pd.DataFrame(model_results)

display(
    model_results_df.sort_values(
        by="Accuracy",
        ascending=False,
    ).reset_index(drop=True)
)

,Model,Accuracy
0,Logistic Regression,0.536892
1,Decision Tree,0.450549
2,Random Forest,0.417582


In [72]:
model_results_df["Accuracy"] = model_results_df["Accuracy"].round(3)

display(model_results_df)

,Model,Accuracy
0,Logistic Regression,0.537
1,Decision Tree,0.451
2,Random Forest,0.418


# Preliminary Model Performance
Three supervised machine learning algorithms were trained using identical preprocessing steps and the same training and testing datasets. Logistic Regression achieved the highest preliminary accuracy (53.7%), followed by Decision Tree (45.1%) and Random Forest (41.8%).

These preliminary results indicate that a linear classification model provided the strongest predictive performance on this dataset. Additional evaluation metrics, including precision, recall, F1-score, confusion matrices, and feature importance, will be presented in the Model Evaluation notebook.

# Save Trained Models

In [73]:
import joblib

MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(logistic_pipeline, MODELS_DIR / "logistic_regression.pkl")
joblib.dump(decision_tree_pipeline, MODELS_DIR / "decision_tree.pkl")
joblib.dump(random_forest_pipeline, MODELS_DIR / "random_forest.pkl")

print("Models saved successfully.")

Models saved successfully.
